# Chapter 05 — Model strategy and customization trade-offs · Practice

**Goal:** turn each model decision from the chapter into a small, testable piece of code: diagnose a failure to the right lever, pick a model configuration from eval runs, check a cascade, run one eval through a Claude and an OpenAI adapter, split data without leakage, size a LoRA adapter, validate a training file, cost a fine-tune, and run a release radar. The mini-project combines them into a **model-strategy decision record** for a client.

- Theory: [README.md](README.md) (read section 5.x before running the matching section here)
- Estimated time: 2–3 hours
- Estimated API cost: **$0** offline. The three optional live cells cost well under $0.05 together.
- Prerequisites: [00-prerequisites](../../00-prerequisites/README.md) (venv from `requirements.txt`; `.env` with `ANTHROPIC_API_KEY` and/or `OPENAI_API_KEY` only for live cells)

| Marker | Meaning |
|---|---|
| **(offline)** | Runs with no key and no network. The Claude and OpenAI cells use **real** `anthropic` / `openai` SDK clients whose HTTP layer is replaced by a local function, so request building and response parsing are the SDKs' own. Eval numbers are illustrative data shaped like a real run. |
| **(live, costs money)** | Calls a real API. It runs only when you set `RUN_LIVE=1` in `.env` **and** the matching key is present; otherwise it prints a skip message. |

Run the notebook top to bottom with a fresh kernel.

In [ ]:
import hashlib
import json
import os
import re
from dataclasses import asdict, dataclass
from datetime import date

import anthropic
import httpx2
import openai
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))          # keys come from .env, never from code
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
CLAUDE_FAST = "claude-haiku-4-5"
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
LIVE = os.getenv("RUN_LIVE") == "1"            # opt-in: live cells cost money
HAVE_CLAUDE = LIVE and bool(os.getenv("ANTHROPIC_API_KEY"))
HAVE_OPENAI = LIVE and bool(os.getenv("OPENAI_API_KEY"))

# USD per million tokens (input, output), standard rates as listed in this repo's vendor chapters.
PRICES = {
    "claude-fable-5-1": (10.00, 50.00), "claude-opus-5-5": (4.00, 20.00),
    "claude-sonnet-5-5": (2.00, 10.00), "claude-haiku-4-5": (1.00, 5.00),
    "gpt-6.1-sol": (2.00, 10.00), "gpt-6-luna": (0.10, 0.50),
}


def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    p_in, p_out = PRICES[model]
    return (input_tokens * p_in + output_tokens * p_out) / 1_000_000


print("anthropic", anthropic.__version__, "| openai", openai.__version__,
      "| live Claude:", HAVE_CLAUDE, "| live OpenAI:", HAVE_OPENAI)

### Offline toolkit and shared data (run once)

`mock_claude(reply_fn)` and `mock_openai(reply_fn)` return real SDK clients. Each request body goes to `reply_fn(body)`, which returns the reply text, a `(text, truncated)` pair, or an HTTP status code (`529`, `500`) to simulate an outage. Token counts are a rough `len // 4` estimate; real counts come from the API's `usage`.

`TICKETS` is a small labeled support-ticket set (fictional company, fictional customers) used across the sections.

In [ ]:
def _tok(s: str) -> int:
    return len(s) // 4 + 1


def _unpack(out):
    if isinstance(out, int):
        return None, False, out
    text, truncated = out if isinstance(out, tuple) else (out, False)
    return text, truncated, 200


def mock_claude(reply_fn) -> anthropic.Anthropic:
    def handle(request: httpx2.Request) -> httpx2.Response:
        body = json.loads(request.content)
        handle.requests.append(body)
        text, truncated, status = _unpack(reply_fn(body))
        if status != 200:
            return httpx2.Response(status, json={"type": "error", "error": {
                "type": "overloaded_error", "message": "offline outage"}})
        return httpx2.Response(200, json={
            "id": "msg_offline", "type": "message", "role": "assistant", "model": body["model"],
            "content": [{"type": "text", "text": text}],
            "stop_reason": "max_tokens" if truncated else "end_turn", "stop_sequence": None,
            "usage": {"input_tokens": _tok(json.dumps(body)), "output_tokens": _tok(text)}})
    handle.requests = []
    client = anthropic.Anthropic(api_key="offline", max_retries=0, http_client=anthropic.DefaultHttpxClient(
        transport=httpx2.MockTransport(handle)))
    client.requests = handle.requests              # inspect what the SDK sent
    return client


def mock_openai(reply_fn) -> openai.OpenAI:
    def handle(request: httpx2.Request) -> httpx2.Response:
        body = json.loads(request.content)
        handle.requests.append(body)
        text, truncated, status = _unpack(reply_fn(body))
        if status != 200:
            return httpx2.Response(status, json={"error": {"message": "offline outage", "type": "server_error"}})
        return httpx2.Response(200, json={
            "id": "resp_offline", "object": "response", "created_at": 0, "model": body["model"],
            "status": "incomplete" if truncated else "completed",
            "incomplete_details": {"reason": "max_output_tokens"} if truncated else None,
            "output": [{"type": "message", "id": "msg_offline", "status": "completed", "role": "assistant",
                        "content": [{"type": "output_text", "text": text, "annotations": []}]}],
            "parallel_tool_calls": True, "tool_choice": "auto", "tools": [],
            "usage": {"input_tokens": _tok(json.dumps(body)), "output_tokens": _tok(text),
                      "total_tokens": 0, "input_tokens_details": {"cached_tokens": 0},
                      "output_tokens_details": {"reasoning_tokens": 0}}})
    handle.requests = []
    client = openai.OpenAI(api_key="offline", max_retries=0, http_client=openai.DefaultHttpxClient(
        transport=httpx2.MockTransport(handle)))
    client.requests = handle.requests
    return client


LABELS = ["billing", "technical", "account", "other"]
TICKETS = [
    {"id": "t01", "customer": "c-ana", "text": "I was charged twice for October.", "label": "billing"},
    {"id": "t02", "customer": "c-ana", "text": "Charged twice again this month, please refund.", "label": "billing"},
    {"id": "t03", "customer": "c-ben", "text": "The router drops Wi-Fi every evening.", "label": "technical"},
    {"id": "t04", "customer": "c-ben", "text": "Wi-Fi keeps dropping after the firmware update.", "label": "technical"},
    {"id": "t05", "customer": "c-cho", "text": "How do I change the email on my account?", "label": "account"},
    {"id": "t06", "customer": "c-cho", "text": "Please add my partner as a second account holder.", "label": "account"},
    {"id": "t07", "customer": "c-dev", "text": "Why is there a late fee on my invoice?", "label": "billing"},
    {"id": "t08", "customer": "c-dev", "text": "Do you sponsor the local football club?", "label": "other"},
    {"id": "t09", "customer": "c-eli", "text": "My SIM shows 'no service' since this morning.", "label": "technical"},
    {"id": "t10", "customer": "c-eli", "text": "I want to close my account at the end of the month.", "label": "account"},
    {"id": "t11", "customer": "c-fay", "text": "The invoice total does not match my plan price.", "label": "billing"},
    {"id": "t12", "customer": "c-fay", "text": "Reset my password, the link has expired.", "label": "account"},
    {"id": "t13", "customer": "c-gus", "text": "Streaming buffers on every device at home.", "label": "technical"},
    {"id": "t14", "customer": "c-gus", "text": "Is your shop open on public holidays?", "label": "other"},
    {"id": "t15", "customer": "c-hal", "text": "Refund the roaming charge, I was never abroad.", "label": "billing"},
    {"id": "t16", "customer": "c-hal", "text": "Port my number from my old provider.", "label": "account"},
]
print(len(TICKETS), "tickets,", len({t['customer'] for t in TICKETS}), "customers")

## 5.1 The customization ladder: knowledge vs behavior

Two kinds of failure, two kinds of fix: missing **knowledge** goes into the context (retrieval, tools), wrong **behavior** goes up the ladder (prompt → model and effort → fine-tuning), one rung at a time. [Theory](README.md#51-the-customization-ladder-knowledge-vs-behavior)

### Diagnose failures to the next rung (offline)

Each failure comes from error analysis on an eval run. `next_rung()` encodes the ladder: knowledge always goes to context; behavior climbs only after the cheaper rung was tried and measured.

In [ ]:
FAILURES = [
    {"id": "F1", "symptom": "Quotes last quarter's refund window", "cause": "knowledge",
     "prompt_tried": True, "effort_swept": True, "volume_high": True},
    {"id": "F2", "symptom": "Answers in long paragraphs; style guide asks for 3 sentences", "cause": "behavior",
     "prompt_tried": False, "effort_swept": False, "volume_high": True},
    {"id": "F3", "symptom": "Misses the second issue in multi-issue tickets", "cause": "behavior",
     "prompt_tried": True, "effort_swept": False, "volume_high": False},
    {"id": "F4", "symptom": "Label set drifts at 2M tickets/day even with a 3k-token few-shot prompt", "cause": "behavior",
     "prompt_tried": True, "effort_swept": True, "volume_high": True},
    {"id": "F5", "symptom": "Does not know the customer's current plan", "cause": "knowledge",
     "prompt_tried": False, "effort_swept": False, "volume_high": False},
]


def next_rung(f: dict) -> str:
    if f["cause"] == "knowledge":
        return "context: retrieval or a tool (never fine-tuning for facts)"
    if not f["prompt_tried"]:
        return "prompt: exact instructions, examples, output schema"
    if not f["effort_swept"]:
        return "model and effort: sweep effort, then the next tier up"
    if f["volume_high"]:
        return "fine-tuning candidate: SFT on curated examples, against a frozen baseline"
    return "error analysis: the failure is not understood yet"


for f in FAILURES:
    print(f"{f['id']} {f['symptom'][:58]:60} -> {next_rung(f)}")

> **Observe:** F1 already tried prompting and effort, and it still goes to context: no amount of behavior work adds a fact. F4 is the only fine-tuning candidate, and even then the next step is a baseline, not a training job. On the Claude API there is no fine-tuning at all (README 5.8), so F4 would mean a shorter prompt plus a cascade, or an open-weight model.

> **Try it:** add a failure "Uses the wrong currency for EU customers" and decide whether it is knowledge (the customer's region is missing from the context) or behavior (the rule is missing from the prompt). Most real failures need that question answered before any lever is pulled.

## 5.2 Eval-driven model selection

Write the bar first, sweep effort before switching models, and choose on **cost per successful task** within the latency budget. [Theory](README.md#52-eval-driven-model-selection)

### Pick a configuration from an eval sweep (offline)

`SWEEP` has the shape of an exported eval run on 200 labeled tickets: pass count, average tokens per item and p95 latency per configuration. The numbers are illustrative; replace them with your own run.

In [ ]:
@dataclass
class Run:
    config: str
    model: str
    passed: int
    total: int
    avg_in: int        # average input tokens per item
    avg_out: int       # average output tokens per item (effort mostly moves this)
    p95_ms: float

    @property
    def cost_usd(self) -> float:   # whole eval run
        return self.total * cost_usd(self.model, self.avg_in, self.avg_out)


SWEEP = [
    Run("opus-5.5 effort=medium", "claude-opus-5-5", 194, 200, 900, 420, 5200),
    Run("opus-5.5 effort=low", "claude-opus-5-5", 191, 200, 900, 180, 2600),
    Run("sonnet-5.5 effort=high", "claude-sonnet-5-5", 190, 200, 900, 520, 4100),
    Run("sonnet-5.5 effort=low", "claude-sonnet-5-5", 187, 200, 900, 150, 1900),
    Run("haiku-4.5", "claude-haiku-4-5", 176, 200, 900, 60, 900),
    Run("gpt-6.1-sol effort=medium", "gpt-6.1-sol", 189, 200, 900, 380, 3400),
    Run("gpt-6-luna effort=low", "gpt-6-luna", 185, 200, 900, 120, 1200),
]


def pick_config(runs: list[Run], bar=0.90, p95_budget_ms=4000):
    ok = [r for r in runs if r.passed / r.total >= bar and r.p95_ms <= p95_budget_ms]
    if not ok:
        return None
    return min(ok, key=lambda r: r.cost_usd / r.passed)


print(f"{'config':27} {'pass':>6} {'p95 ms':>7} {'$/call':>9} {'$/success':>10}")
for r in SWEEP:
    print(f"{r.config:27} {r.passed / r.total:6.1%} {r.p95_ms:7.0f} {r.cost_usd / r.total:9.5f} {r.cost_usd / r.passed:10.5f}")

for bar, budget in [(0.92, 3000), (0.96, 3000), (0.96, 6000)]:
    best = pick_config(SWEEP, bar, budget)
    print(f"bar={bar:.0%} p95<={budget}ms ->", best.config if best else "None: fix prompt/context first")

> **Observe:** at a 92% bar and a 3 s budget, three configurations qualify and the cheapest per success wins by a wide margin. Raise the bar to 96% and nothing qualifies inside 3 s: that is a signal to look at the failures, not to buy the biggest model. Only a relaxed latency budget lets Opus 5.5 at `medium` in. Effort mattered as much as the model: Opus at `low` fits the 3 s budget, Opus at `medium` does not.

> **Try it:** the winner is on a different vendor than the client's current one. List what you would still check before recommending it (README 5.2 step 5: residency, the platform the client buys through, structured-output support, retirement date). A cheaper row is a candidate, not a decision.

In [ ]:
# (live, costs money) a tiny effort sweep on the real API: 8 tickets x 2 effort levels on Sonnet 5.5
SYSTEM = f"Classify the support ticket. Reply with exactly one word from: {', '.join(LABELS)}."
if not HAVE_CLAUDE:
    print("skipped: set RUN_LIVE=1 and ANTHROPIC_API_KEY in .env to run this cell")
else:
    live = anthropic.Anthropic()
    for effort in ["low", "high"]:
        passed, spend = 0, 0.0
        for t in TICKETS[:8]:
            r = live.messages.create(model=CLAUDE_MODEL, max_tokens=1024, output_config={"effort": effort},
                                     system=SYSTEM, messages=[{"role": "user", "content": t["text"]}])
            text = "".join(b.text for b in r.content if b.type == "text").strip().lower()
            passed += text == t["label"]
            spend += cost_usd(CLAUDE_MODEL, r.usage.input_tokens, r.usage.output_tokens)
        print(f"effort={effort:5} passed {passed}/8, ${spend:.4f}, ${spend / max(passed, 1):.5f} per success")

## 5.3 Multi-model designs: routing, cascades, advisors and orchestrators

A pairing must beat one model at its best effort setting. A cascade is only as good as its gate. [Theory](README.md#53-multi-model-designs-routing-cascades-advisors-and-orchestrators)

### Cascade vs single model, with a weak and a strong gate (offline)

Each eval item records what the cheap model did (`ok`, `invalid` = fails schema or a code check, `wrong` = well-formed but wrong) and whether the strong model got it right. Escalated items pay for both calls.

In [ ]:
def outcomes(n_ok, n_invalid, n_wrong, strong_misses):
    items = [{"cheap": "ok"}] * n_ok + [{"cheap": "invalid"}] * n_invalid + [{"cheap": "wrong"}] * n_wrong
    return [dict(it, strong_ok=i not in strong_misses) for i, it in enumerate(items)]


ITEMS = outcomes(n_ok=170, n_invalid=18, n_wrong=12, strong_misses={171, 195, 198})
C_CHEAP, C_STRONG = 0.0004, 0.0060          # $ per item for each model on this task


def cascade_report(items, gate_catches_wrong=0.0):
    correct = cost = silent = escalated = 0
    wrong_seen = 0
    for it in items:
        cost += C_CHEAP
        escalate = it["cheap"] == "invalid"
        if it["cheap"] == "wrong":           # a business-rule gate catches a share of these
            wrong_seen += 1
            escalate = wrong_seen <= round(gate_catches_wrong * sum(i["cheap"] == "wrong" for i in items))
        if escalate:
            escalated += 1
            cost += C_STRONG
            correct += it["strong_ok"]
        else:
            correct += it["cheap"] == "ok"
            silent += it["cheap"] == "wrong"
    n = len(items)
    return {"accuracy": correct / n, "cost_per_item": cost / n, "escalation_rate": escalated / n, "silent_wrong": silent}


all_strong = {"accuracy": sum(i["strong_ok"] for i in ITEMS) / len(ITEMS), "cost_per_item": C_STRONG}
print("all strong          :", all_strong)
print("cascade, schema gate:", cascade_report(ITEMS))
print("cascade, + rule gate:", cascade_report(ITEMS, gate_catches_wrong=0.75))

> **Observe:** the schema-only cascade is about 6× cheaper than all-strong but leaves 12 well-formed wrong answers that nothing escalates. A business-rule gate (for example "a `billing` label needs an amount or an invoice reference in the ticket") catches most of them for a small extra cost. Check the cascade's **accuracy** on the eval, not just its cost.

> **Try it:** compare the cascade with a *single* model at a lower effort. If `sonnet-5.5 effort=low` from 5.2 clears the bar at $0.0033 per item with no gate code to maintain, is the cascade still worth its complexity? Anthropic's guidance says a pairing has to beat that whole single-model curve.

## 5.4 Provider-agnostic design and lock-in

A thin adapter around the one thing every vendor does; prompts, schemas, routing and evals stay in shared code. [Theory](README.md#54-provider-agnostic-design-and-lock-in)

### Two adapters, one eval (offline, real SDK clients)

The adapters are the README's. The "model" behind both mocks is a keyword rule, so the point here is the wiring: the same `run_eval()` scores either vendor, and the request bodies show what each SDK actually sends.

In [ ]:
@dataclass
class Reply:
    text: str
    input_tokens: int
    output_tokens: int
    truncated: bool


class ClaudeChat:
    def __init__(self, client, model): self.c, self.m = client, model

    def complete(self, system: str, user: str, max_tokens=1024) -> Reply:
        r = self.c.messages.create(model=self.m, max_tokens=max_tokens, system=system,
                                   messages=[{"role": "user", "content": user}])
        text = "".join(b.text for b in r.content if b.type == "text")
        return Reply(text, r.usage.input_tokens, r.usage.output_tokens, r.stop_reason == "max_tokens")


class OpenAIChat:
    def __init__(self, client, model): self.c, self.m = client, model

    def complete(self, system: str, user: str, max_tokens=1024) -> Reply:
        r = self.c.responses.create(model=self.m, instructions=system, input=user,
                                    max_output_tokens=max_tokens, store=False)
        return Reply(r.output_text, r.usage.input_tokens, r.usage.output_tokens, r.status == "incomplete")


KEYWORDS = {"billing": ["charge", "invoice", "refund", "fee"], "technical": ["wi-fi", "router", "sim", "buffer"],
            "account": ["account", "password", "email", "port"]}


def keyword_model(text: str) -> str:
    low = text.lower()
    return next((lab for lab, words in KEYWORDS.items() if any(w in low for w in words)), "other")


def claude_reply(body):
    return keyword_model(body["messages"][-1]["content"])


def openai_reply(body):
    return keyword_model(body["input"])


SYSTEM = f"Classify the support ticket. Reply with exactly one word from: {', '.join(LABELS)}."


def run_eval(chat, model: str, items) -> dict:
    passed, spend = 0, 0.0
    for t in items:
        r = chat.complete(SYSTEM, t["text"], max_tokens=16)
        passed += (not r.truncated) and r.text.strip().lower() == t["label"]
        spend += cost_usd(model, r.input_tokens, r.output_tokens)
    return {"pass_rate": round(passed / len(items), 3), "cost_usd": round(spend, 6)}


fake_c, fake_o = mock_claude(claude_reply), mock_openai(openai_reply)
print("Claude adapter:", run_eval(ClaudeChat(fake_c, CLAUDE_MODEL), CLAUDE_MODEL, TICKETS))
print("OpenAI adapter:", run_eval(OpenAIChat(fake_o, OPENAI_MODEL), OPENAI_MODEL, TICKETS))
print("Claude request keys:", sorted(fake_c.requests[0]))
print("OpenAI request keys:", sorted(fake_o.requests[0]))

> **Observe:** the system prompt travels as `system` to Claude and as `instructions` to OpenAI, and the stop signal is `stop_reason == "max_tokens"` vs `status == "incomplete"`. The adapter hides those differences; the eval, labels and prompt live outside it. That is why a vendor switch becomes "re-run the eval and compare", not a rewrite.

### Failover between vendors (offline)

A second vendor is also a resilience option. `FallbackChat` tries the primary and moves to the secondary on a server-side error. It only makes sense if the secondary's **own prompt** has passed the eval, so it takes its own system prompt.

In [ ]:
class FallbackChat:
    def __init__(self, primary, secondary, secondary_system: str):
        self.primary, self.secondary, self.secondary_system = primary, secondary, secondary_system
        self.used = []

    def complete(self, system: str, user: str, max_tokens=1024) -> Reply:
        try:
            reply = self.primary.complete(system, user, max_tokens)
            self.used.append("primary")
            return reply
        except (anthropic.APIStatusError, openai.APIStatusError) as exc:
            if exc.status_code < 500:              # a 4xx is our bug: do not hide it behind a failover
                raise
            self.used.append(f"secondary after {exc.status_code}")
            return self.secondary.complete(self.secondary_system, user, max_tokens)


calls = {"n": 0}


def flaky_claude(body):
    calls["n"] += 1
    return 529 if calls["n"] % 3 == 0 else claude_reply(body)   # every third call: overloaded


chat = FallbackChat(ClaudeChat(mock_claude(flaky_claude), CLAUDE_MODEL),
                    OpenAIChat(mock_openai(openai_reply), OPENAI_MODEL), secondary_system=SYSTEM)
answers = [chat.complete(SYSTEM, t["text"], 16).text for t in TICKETS[:6]]
print(answers)
print(chat.used)

In [ ]:
# (live, costs money) the same eval on both real APIs, 6 tickets each
if not (HAVE_CLAUDE or HAVE_OPENAI):
    print("skipped: set RUN_LIVE=1 and at least one API key in .env to run this cell")
if HAVE_CLAUDE:
    print("Claude:", run_eval(ClaudeChat(anthropic.Anthropic(), CLAUDE_FAST), CLAUDE_FAST, TICKETS[:6]))
if HAVE_OPENAI:
    print("OpenAI:", run_eval(OpenAIChat(openai.OpenAI(), OPENAI_MODEL), OPENAI_MODEL, TICKETS[:6]))

## 5.5 Just enough ML: how models learn and how to measure them

Your eval set is the test set: split it honestly and pick metrics by the cost of each error. [Theory](README.md#55-just-enough-ml-how-models-learn-and-how-to-measure-them)

### Precision, recall and class imbalance (offline)

In [ ]:
def prf(y_true: list[str], y_pred: list[str], positive: str) -> dict:
    tp = sum(t == positive and p == positive for t, p in zip(y_true, y_pred))
    fp = sum(t != positive and p == positive for t, p in zip(y_true, y_pred))
    fn = sum(t == positive and p != positive for t, p in zip(y_true, y_pred))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)}


# 1,000 transactions, 2% fraud. A "model" that never flags fraud:
truth = ["fraud"] * 20 + ["ok"] * 980
never = ["ok"] * 1000
accuracy = sum(t == p for t, p in zip(truth, never)) / len(truth)
print(f"never-fraud accuracy {accuracy:.0%}, fraud metrics {prf(truth, never, 'fraud')}")

# A model that flags 30 items and catches 16 of the 20 frauds:
flags = ["fraud"] * 16 + ["ok"] * 4 + ["fraud"] * 14 + ["ok"] * 966
print("flagging model fraud metrics", prf(truth, flags, "fraud"))

### Leakage: split by row vs split by group (offline)

Several tickets come from the same customer and look alike. A row split puts near-copies on both sides; a group split keeps each customer on one side. The hash makes the split stable across runs and machines.

In [ ]:
def split_by_group(rows, key, test_pct=20):
    def bucket(g):
        return int(hashlib.sha256(g.encode()).hexdigest(), 16) % 100
    test_groups = {r[key] for r in rows if bucket(r[key]) < test_pct}
    return ([r for r in rows if r[key] not in test_groups], [r for r in rows if r[key] in test_groups])


def shared_groups(train, test, key) -> set:
    return {r[key] for r in train} & {r[key] for r in test}


row_train, row_test = TICKETS[::2] + TICKETS[1::4], TICKETS[3::4]      # a naive row split
grp_train, grp_test = split_by_group(TICKETS, "customer", test_pct=30)
print("row split   : customers on both sides =", sorted(shared_groups(row_train, row_test, "customer")))
print("group split : customers on both sides =", sorted(shared_groups(grp_train, grp_test, "customer")),
      f"| test customers: {sorted({r['customer'] for r in grp_test})}")

### Overfitting: pick the epoch from validation loss (offline)

A training log from a fine-tuning run. Training loss keeps falling; validation loss turns up when the model starts to memorize.

In [ ]:
LOG = [(1, 1.92, 1.95), (2, 1.31, 1.42), (3, 0.88, 1.18), (4, 0.52, 1.21), (5, 0.27, 1.37), (6, 0.12, 1.58)]
best_epoch, _, best_val = min(LOG, key=lambda row: row[2])
for epoch, tr, va in LOG:
    print(f"epoch {epoch}: train {tr:.2f}  val {va:.2f}" + ("  <- best" if epoch == best_epoch else ""))
print(f"Stop at epoch {best_epoch}; later epochs fit the training set better and new data worse.")

## 5.6 Fine-tuning methods and parameter-efficient tuning (LoRA)

Pick the method from the shape of your data, and know why adapters change the hosting picture. [Theory](README.md#56-fine-tuning-methods-and-parameter-efficient-tuning-lora)

### Size a LoRA adapter (offline)

An illustrative 8-billion-parameter open-weight model with 32 layers and hidden size 4,096. LoRA is applied to the four attention projections in each layer (query, key, value, output). Weights stored at 2 bytes each (bf16).

Look for: at rank 16 the adapter is a few tens of megabytes, a fraction of a percent of the base, so 40 tenant variants cost little more than one base model. That only applies to weights you host; a hosted vendor fine-tune gives you a model ID instead. The second cell maps the shape of your data to a method (README 5.6 table).

In [ ]:
def lora_params(d_in: int, d_out: int, rank: int) -> int:
    return rank * (d_in + d_out)


LAYERS, HIDDEN, PROJECTIONS, BASE_PARAMS, BYTES = 32, 4096, 4, 8_000_000_000, 2
for rank in (8, 16, 64):
    trainable = LAYERS * PROJECTIONS * lora_params(HIDDEN, HIDDEN, rank)
    print(f"rank {rank:2}: {trainable:>11,} trainable params = {trainable / BASE_PARAMS:.3%} of the base, "
          f"adapter file ~{trainable * BYTES / 1e6:,.0f} MB")

tenants = 40
adapter_mb = LAYERS * PROJECTIONS * lora_params(HIDDEN, HIDDEN, 16) * BYTES / 1e6
print(f"\n{tenants} tenants: full fine-tunes ~{tenants * BASE_PARAMS * BYTES / 1e9:,.0f} GB "
      f"vs one base + {tenants} adapters ~{(BASE_PARAMS * BYTES / 1e6 + tenants * adapter_mb) / 1e3:,.1f} GB")

In [ ]:
def choose_method(data: str, gradeable: bool = False, teacher_good: bool = False) -> str:
    # data: "pairs" (input -> ideal output), "preferences" (chosen vs rejected), "inputs_only"
    if data == "preferences":
        return "DPO (preference tuning)"
    if data == "inputs_only" and gradeable:
        return "RFT (reinforcement fine-tuning with a grader)"
    if data == "inputs_only" and teacher_good:
        return "distillation: label inputs with the strong model, then SFT a small one"
    if data == "pairs":
        return "SFT (supervised fine-tuning)"
    return "not a fine-tuning case yet: collect labeled data first"


for case in [("pairs",), ("preferences",), ("inputs_only", True), ("inputs_only", False, True), ("inputs_only",)]:
    print(f"{str(case):32} -> {choose_method(*case)}")

## 5.7 Training data: curation, formats and governance

Validate, dedupe, scrub and document the data before anything is uploaded; keep the test set out of training. [Theory](README.md#57-training-data-curation-formats-and-governance)

### Build SFT lines from labeled tickets and validate them (offline)

In [ ]:
def to_sft(ticket: dict) -> str:
    return json.dumps({"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": ticket["text"]},
        {"role": "assistant", "content": ticket["label"]}]})


def check_sft_line(line: str) -> list[str]:
    try:
        ex = json.loads(line)
    except json.JSONDecodeError:
        return ["not valid JSON"]
    msgs = ex.get("messages") if isinstance(ex, dict) else None
    if not msgs:
        return ["missing 'messages'"]
    errs = []
    roles = [m.get("role") for m in msgs]
    if roles[-1] != "assistant":
        errs.append("last message must be the assistant's target")
    if any(r not in {"system", "user", "assistant", "tool"} for r in roles):
        errs.append(f"unexpected role in {roles}")
    if any(not str(m.get("content", "")).strip() for m in msgs):
        errs.append("empty content")
    return errs


def check_dpo_line(line: str) -> list[str]:
    try:
        ex = json.loads(line)
    except json.JSONDecodeError:
        return ["not valid JSON"]
    errs = []
    turns = ex.get("input", {}).get("messages", [])
    if not turns or turns[-1].get("role") != "user":
        errs.append("input.messages must end with a user turn (single-turn input)")
    for side in ("preferred_output", "non_preferred_output"):
        out = ex.get(side) or [{}]
        if out[-1].get("role") != "assistant" or not str(out[-1].get("content", "")).strip():
            errs.append(f"{side} must end with a non-empty assistant message")
    if ex.get("preferred_output") == ex.get("non_preferred_output"):
        errs.append("preferred and non-preferred answers are identical")
    return errs


lines = [to_sft(t) for t in grp_train]
lines += ['{"messages": [{"role": "user", "content": "Refund me"}]}', "not json"]
for i, line in enumerate(lines):
    if problems := check_sft_line(line):
        print(f"line {i}: {problems}")
dpo = json.dumps({"input": {"messages": [{"role": "user", "content": "Summarize the call"}]},
                  "preferred_output": [{"role": "assistant", "content": "Customer wants a refund for a double charge."}],
                  "non_preferred_output": [{"role": "assistant", "content": "A call happened."}]})
print("DPO line problems:", check_dpo_line(dpo) or "none")

### Dedupe, scrub PII, check test overlap and write a data card (offline)

In [ ]:
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
PHONE = re.compile(r"\+?\d[\d ()-]{7,}\d")


def scrub(text: str) -> str:
    return PHONE.sub("<PHONE>", EMAIL.sub("<EMAIL>", text))


def norm(text: str) -> str:
    return re.sub(r"\W+", " ", text.lower()).strip()


raw = [t["text"] for t in grp_train] + ["I was charged TWICE for October!", "Call me on +44 20 7946 0958 or ana@example.com"]
seen, clean = set(), []
for text in raw:
    key = norm(text)
    if key not in seen:
        seen.add(key)
        clean.append(scrub(text))
print(f"{len(raw)} raw -> {len(clean)} after dedupe; last: {clean[-1]!r}")

test_keys = {norm(t["text"]) for t in grp_test}
print("train/test overlap:", [c for c in clean if norm(c) in test_keys] or "none")

train_file = "\n".join(to_sft(t) for t in grp_train)
DATA_CARD = {
    "name": "ticket-triage-sft", "source": "support tickets, fictional sample", "rows": len(grp_train),
    "labels": LABELS, "label_guideline": "v2", "split": "by customer, sha256 bucket, 30% test",
    "pii": "emails and phone numbers replaced with tokens", "known_gaps": "no non-English tickets",
    "legal_basis": "to be confirmed by the client's DPO", "sha256": hashlib.sha256(train_file.encode()).hexdigest()[:16],
}
print(json.dumps(DATA_CARD, indent=1))

## 5.8 The fine-tuning lifecycle, hosting and total cost

Version every artifact, compare with the prompt-only baseline on the same test set, and count every cost, including the re-tune when the base model retires. [Theory](README.md#58-the-fine-tuning-lifecycle-hosting-and-total-cost)

### A registry record and a release gate (offline)

In [ ]:
@dataclass(frozen=True)
class TunedModel:
    name: str
    base_model: str
    method: str
    data_sha256: str
    test_set: str
    score: float
    baseline_score: float
    regression_score: float      # general-skills set the model must not get worse at
    baseline_regression: float
    base_retires: str | None


def release_gate(m: TunedModel, min_gain=0.02, max_regression_drop=0.01) -> list[str]:
    problems = []
    if m.score - m.baseline_score < min_gain:
        problems.append(f"gain {m.score - m.baseline_score:+.3f} below {min_gain}")
    if m.baseline_regression - m.regression_score > max_regression_drop:
        problems.append(f"general skills dropped {m.baseline_regression - m.regression_score:.3f}")
    if m.base_retires is None:
        problems.append("base model retirement date unknown: plan the re-tune")
    return problems


candidate = TunedModel("ticket-triage-lora-v3", "open-weight-8b-2026-06", "lora-sft", DATA_CARD["sha256"],
                       "triage-test-2026-10", score=0.951, baseline_score=0.934,
                       regression_score=0.880, baseline_regression=0.905, base_retires=None)
print(json.dumps(asdict(candidate), indent=1))
print("release gate:", release_gate(candidate) or "pass")

### Does the fine-tune pay for itself? (offline)

Compare the prompt-only design (a long few-shot prompt on a mid-tier model) with a fine-tuned small model that needs a short prompt but adds hosting, training, labeling and a re-tune whenever its base retires.

In [ ]:
def monthly_cost_prompt_only(calls, in_tok=3200, out_tok=20, model="claude-sonnet-5-5"):
    return calls * cost_usd(model, in_tok, out_tok)


def monthly_cost_tuned(calls, per_call=0.00004, hosting=2_500.0, one_off=18_000.0, retune=9_000.0,
                       retunes_per_year=1, amortize_months=12):
    return calls * per_call + hosting + one_off / amortize_months + retune * retunes_per_year / 12


print(f"{'calls/month':>12} {'prompt-only':>12} {'fine-tuned':>11}  winner")
for calls in (100_000, 1_000_000, 10_000_000, 60_000_000):
    a, b = monthly_cost_prompt_only(calls), monthly_cost_tuned(calls)
    print(f"{calls:>12,} {a:>12,.0f} {b:>11,.0f}  {'fine-tuned' if b < a else 'prompt-only'}")

> **Observe:** below roughly 700,000 calls a month the fixed costs (hosting, labeling, the yearly re-tune) dominate and the prompt-only design wins. At high volume the shorter prompt and smaller model pay off, which is the case 5.1 describes. Before trusting the crossover, check the cheaper alternatives the chapter lists: prompt caching on the long prefix, a smaller model with a better prompt, or a cascade.

## 5.9 Staying current: a release radar

Watch the vendors' lifecycle pages, trigger an eval on every relevant event, and recommend in writing with a date. [Theory](README.md#59-staying-current-a-release-radar)

### Radar over the models a client uses (offline, real lifecycle data)

Statuses and dates from the Claude model deprecations page, checked 2026-11-06 (`committed` = the "not sooner than" retirement date). Re-check before you rely on them.

In [ ]:
TODAY = date(2026, 11, 6)
LIFECYCLE = {   # model -> (status, retire_date, committed_until)
    "claude-sonnet-4-5-20250929": ("deprecated", date(2026, 11, 30), None),
    "claude-haiku-4-5-20251001": ("active", None, date(2026, 10, 15)),
    "claude-opus-4-5-20251101": ("active", None, date(2026, 11, 24)),
    "claude-sonnet-5-5": ("active", None, date(2027, 9, 28)),
    "claude-haiku-5-5": ("active", None, date(2027, 10, 7)),
}


def radar(in_use: dict, today: date, window_days=90) -> list[str]:
    alerts = []
    for model, (status, retires, committed) in in_use.items():
        if status == "deprecated":
            alerts.append(f"{model}: deprecated, retires {retires} ({(retires - today).days} days)")
        elif committed and committed <= today:
            alerts.append(f"{model}: commitment passed; deprecation can come with 60 days' notice")
        elif committed and (committed - today).days <= window_days:
            alerts.append(f"{model}: commitment ends {committed}")
    return alerts


for alert in radar(LIFECYCLE, TODAY):
    print("-", alert)

In [ ]:
def recommendation(current: Run, candidate: Run, deadline: date, migration_days: int) -> str:
    cur, new = current.cost_usd / current.passed, candidate.cost_usd / candidate.passed
    quality = candidate.passed / candidate.total - current.passed / current.total
    verdict = ("switch" if quality >= 0 and new <= cur else "hold and re-test" if quality < 0
               else "switch only if the quality gain is worth the extra cost")
    return (f"Recommendation for {current.config} -> {candidate.config}: {verdict}.\n"
            f"- Quality on the client eval: {quality:+.1%} pass rate\n"
            f"- Cost per successful task: ${cur:.5f} -> ${new:.5f} ({new / cur - 1:+.0%})\n"
            f"- Migration effort: about {migration_days} days, staged rollout with tested rollback\n"
            f"- Decide by: {deadline.isoformat()}")


print(recommendation(SWEEP[3], SWEEP[1], deadline=date(2026, 11, 20), migration_days=5))

> **Observe:** the radar turns vendor news into owned, dated items, and the recommendation is built from the client's own eval numbers, not the launch announcement. Sonnet 4.5 is 24 days from retirement, so its row is urgent; Haiku 4.5 is active but past its commitment date, so it needs a tested successor ready.

> **Try it:** add `"temperature removed in API version 2026-10-21"` as a radar item for every service that still sends `temperature`, with an owner and a date. API changes break code as surely as retirements do.

## Mini-project: a model-strategy decision record

**Client brief (fictional):** *Northwind Mobile* triages 1.2 million support tickets a month into four queues. Today it runs a 3,200-token few-shot prompt on `claude-sonnet-5-5` at default effort and wants to (a) cut cost, (b) know whether to fine-tune, and (c) avoid being stuck on one vendor. Bar: 92% pass on the 200-item eval, p95 under 3 s. The 5.2 sweep was run after a prompt rewrite that cut the prompt to about 900 tokens, so "today" (the 3,200-token prompt) and the sweep rows are priced differently on purpose.

`decision_record()` below assembles the pieces above: the failure diagnoses (5.1), the sweep (5.2), the cascade report (5.3), the fine-tune cost comparison (5.8) and the radar (5.9). Read it, run it, then run the reviewer checks in the next cell. Extend it with anything your own client would ask for.

In [ ]:
def decision_record(volume=1_200_000) -> str:
    chosen = pick_config(SWEEP, bar=0.92, p95_budget_ms=3000)
    runner_up = min((r for r in SWEEP if r is not chosen and r.model.startswith("claude")
                     and r.passed / r.total >= 0.92 and r.p95_ms <= 3000), key=lambda r: r.cost_usd / r.passed)
    casc = cascade_report(ITEMS, gate_catches_wrong=0.75)
    today_cost = monthly_cost_prompt_only(volume)              # 3,200-token prompt on Sonnet 5.5
    chosen_cost = volume * chosen.cost_usd / chosen.total
    tuned = monthly_cost_tuned(volume)
    lines = [
        "# Decision record: Northwind Mobile ticket triage",
        f"Date: {TODAY.isoformat()} | Bar: 92% pass on 200-item eval, p95 <= 3 s",
        "## Failures and levers",
        *[f"- {f['id']}: {f['symptom']} -> {next_rung(f)}" for f in FAILURES],
        "## Model choice",
        f"- Chosen: {chosen.config} ({chosen.passed / chosen.total:.1%}, ${chosen.cost_usd / chosen.passed:.5f} per success)",
        f"- Current-vendor fallback (Claude): {runner_up.config} ({runner_up.passed / runner_up.total:.1%})",
        f"- Cascade checked: {casc['accuracy']:.1%} accuracy, {casc['silent_wrong']} silent errors; "
        "not adopted while a single model clears the bar",
        "## Fine-tuning",
        f"- At {volume:,} calls/month: today ${today_cost:,.0f}, chosen config ${chosen_cost:,.0f}, "
        f"fine-tuned small model ${tuned:,.0f} (hosting, labeling, re-tunes included)",
        "- Decision: " + ("revisit fine-tuning" if tuned < chosen_cost
                          else "no fine-tune now: model selection already beats it on cost"),
        "- Claude API has no fine-tuning; OpenAI self-serve fine-tuning is closed to new organizations",
        "## Portability",
        "- Thin adapters per vendor; prompts, labels and the eval suite live in the client's repo",
        "## Re-evaluation triggers",
        *[f"- {a}" for a in radar(LIFECYCLE, TODAY)],
        "- Any new model in a tier we use, a price change, or an API version change: re-run the eval",
    ]
    return "\n".join(lines)


record = decision_record()
print(record)

In [ ]:
# Checks a reviewer would make (offline)
checks = {
    "names a chosen config": "Chosen:" in record,
    "chosen config clears the bar": pick_config(SWEEP, 0.92, 3000).config in record,
    "has a fallback": "fallback" in record,
    "addresses fine-tuning with numbers": "fine-tuned small model $" in record,
    "compares against the chosen config, not today's": "no fine-tune now" in record,
    "states platform limits": "no fine-tuning" in record,
    "lists dated re-evaluation triggers": "retires 2026-11-30" in record,
}
for name, ok in checks.items():
    print("✓" if ok else "✗", name)
assert all(checks.values()), "the decision record is missing something a reviewer expects"

> **Try it:** change the client's volume to 60 million calls a month and the bar to 95%. Which lines of the record change, and what would you now ask the client before recommending a fine-tune (data rights, an open-weight host, who owns the re-tune)?

Next: the self-test in [02_homework.ipynb](02_homework.ipynb).